In [11]:
import time
from google.cloud import bigquery
import os

data_path = "../data"
files = os.listdir(data_path)
for file in files:
    print(file)

table_name = input("Enter the name of the table you are about to create: \n")

table_id = f"main-db-495401.mmem_main_dataset.{table_name}"

# Create a BigQuery client
client = bigquery.Client()

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
)

# Open the CSV file and load it into BigQuery
with open(f"{data_path}/{table_name}.csv", "rb") as source_file:
    job = client.load_table_from_file(source_file, table_id, job_config=job_config)

    while job.state != 'DONE':
        time.sleep(2)
        job.reload()
        print(job.state)

print(job.result())

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


DONE
LoadJob<project=main-db-495401, location=northamerica-south1, id=d62e0f5d-e929-4222-83b6-f7758f65df2d>


In [9]:
import io
import time
import pandas as pd
from google.cloud import bigquery

# Special handling for olist_order_reviews_dataset.csv
# review_comment_message contains raw user text with encoding quirks, embedded
# newlines, and mismatched quotes — read with latin-1 to accept any byte value.

reviews_path = "../data/olist_order_reviews_dataset.csv"
reviews_table_id = "main-db-495401.mmem_main_dataset.olist_order_reviews_dataset"

df_reviews = pd.read_csv(
    reviews_path,
    encoding="latin-1",   # accepts every byte value — no UnicodeDecodeError
    on_bad_lines="warn",  # warn on malformed lines without crashing
)

# Write to an in-memory buffer and use load_table_from_file — no pyarrow needed
buffer = io.BytesIO()
df_reviews.to_csv(buffer, index=False, encoding="utf-8")
buffer.seek(0)

client = bigquery.Client()

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    autodetect=True,
    allow_quoted_newlines=True,
)

job = client.load_table_from_file(buffer, reviews_table_id, job_config=job_config)

while job.state != "DONE":
    time.sleep(2)
    job.reload()
    print(job.state)

print(job.result())
print(f"Loaded {client.get_table(reviews_table_id).num_rows} rows into {reviews_table_id}")


DONE
LoadJob<project=main-db-495401, location=northamerica-south1, id=e90e4308-141c-4331-8bae-3313a30ed418>
Loaded 99224 rows into main-db-495401.mmem_main_dataset.olist_order_reviews_dataset
